# Phase 2: source profiling

One sample of every source, profiled for row counts, date range, nulls, duplicate keys and geography codes.
Samples come from `notebooks/download_samples.py` and live under `data/samples/` (git-ignored).
Findings are written up in [`docs/data_dictionary.md`](../docs/data_dictionary.md).

In [1]:
import json
from datetime import date
from pathlib import Path

import duckdb
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
S = (ROOT / "data" / "samples").as_posix()
METROS = yaml.safe_load((ROOT / "config" / "metros.yml").read_text())["metros"]
CBSAS = [m["cbsa_code"] for m in METROS]
TODAY = date.today()

con = duckdb.connect()
for ext in ("excel", "spatial"):
    con.install_extension(ext)
    con.load_extension(ext)
pd.set_option("display.max_rows", 100, "display.max_columns", 40, "display.width", 200)
SUMMARY = []


def q(sql):
    return con.execute(sql).df()


def profile(view, keys, date_col=None, geo=None, vintage=None):
    """Print rows, key duplicates and date range; return per-column type, null % and range."""
    k = ", ".join(keys)
    rows = con.execute(f"select count(*) from {view}").fetchone()[0]
    dups = con.execute(
        f"select count(*) from (select {k} from {view} group by all having count(*) > 1)"
    ).fetchone()[0]
    lo, hi = (
        con.execute(f"select min({date_col}), max({date_col}) from {view}").fetchone()
        if date_col
        else (None, None)
    )
    SUMMARY.append(
        {
            "source": view,
            "rows": rows,
            "key": k,
            "duplicate_keys": dups,
            "date_from": lo,
            "date_to": hi,
            "vintage": vintage,
            "geography": geo,
        }
    )
    print(
        f"{view}: {rows:,} rows | key ({k}): {dups:,} duplicated | "
        + (f"{date_col}: {lo} to {hi}" if date_col else f"vintage: {vintage}")
        + f" | geography: {geo}"
    )
    return q(
        f"select column_name, column_type, null_percentage, approx_unique, min, max from (summarize {view})"
    )

## 1. Zillow (ZHVI home values, ZORI rents)

Wide files: one row per region, one column per month. Unpivoted here to (region, month, value); UNPIVOT drops empty
months, so the fill-rate table below compares rows against regions × months.

In [2]:
for view, f in {
    "zillow_zip_zhvi": "zip_zhvi",
    "zillow_zip_zori": "zip_zori",
    "zillow_metro_zhvi": "metro_zhvi",
    "zillow_metro_zori": "metro_zori",
}.items():
    con.sql(
        f"create or replace table {view}_wide as select * from read_csv('{S}/zillow/{f}.csv', types={{'RegionName': 'VARCHAR'}})"
    )
    con.sql(f"""
        create or replace view {view} as
        select * replace (month::date as month)
        from (unpivot {view}_wide on columns('^\\d{{4}}-\\d{{2}}-\\d{{2}}$') into name month value value)""")
profile(
    "zillow_zip_zhvi", ["RegionID", "month"], "month", "ZIP (RegionName, 5-char text)"
)

zillow_zip_zhvi: 6,508,601 rows | key (RegionID, month): 0 duplicated | month: 2000-01-31 to 2026-09-30 | geography: ZIP (RegionName, 5-char text)


,column_name,column_type,null_percentage,approx_unique,min,max
0,RegionID,BIGINT,0.00,25580,58196,808739
1,SizeRank,BIGINT,0.00,14982,0,33010
2,RegionName,VARCHAR,0.00,25889,01001,99929
3,RegionType,VARCHAR,0.00,1,zip,zip
4,StateName,VARCHAR,0.00,51,AK,WY
5,State,VARCHAR,0.00,51,AK,WY
6,City,VARCHAR,0.02,13107,Aaronsburg,Zwolle
7,Metro,VARCHAR,12.43,952,"Aberdeen, SD","Zapata, TX"
8,CountyName,VARCHAR,0.00,1858,Abbeville County,Zavala County
9,month,DATE,0.00,312,2000-01-31,2026-09-30


In [3]:
profile(
    "zillow_zip_zori", ["RegionID", "month"], "month", "ZIP (RegionName, 5-char text)"
)

zillow_zip_zori: 469,420 rows | key (RegionID, month): 0 duplicated | month: 2015-01-31 to 2026-09-30 | geography: ZIP (RegionName, 5-char text)


,column_name,column_type,null_percentage,approx_unique,min,max
0,RegionID,BIGINT,0.00,8694,58197,870188
1,SizeRank,BIGINT,0.00,7234,0,33010
2,RegionName,VARCHAR,0.00,8345,01002,99801
3,RegionType,VARCHAR,0.00,1,zip,zip
4,StateName,VARCHAR,0.00,51,AK,WY
5,State,VARCHAR,0.00,51,AK,WY
6,City,VARCHAR,0.00,3483,Aberdeen,Zionsville
7,Metro,VARCHAR,0.05,715,"Aberdeen, SD","Zanesville, OH"
8,CountyName,VARCHAR,0.00,894,Ada County,Yuma County
9,month,DATE,0.00,141,2015-01-31,2026-09-30


In [4]:
profile(
    "zillow_metro_zhvi",
    ["RegionID", "month"],
    "month",
    "Zillow metro ID + name, no CBSA code",
)

zillow_metro_zhvi: 238,029 rows | key (RegionID, month): 0 duplicated | month: 2000-01-31 to 2026-09-30 | geography: Zillow metro ID + name, no CBSA code


,column_name,column_type,null_percentage,approx_unique,min,max
0,RegionID,BIGINT,0.00,924,102001,753929
1,SizeRank,BIGINT,0.00,891,0,939
2,RegionName,VARCHAR,0.00,912,"Aberdeen, SD","Zapata, TX"
3,RegionType,VARCHAR,0.00,2,country,msa
4,StateName,VARCHAR,0.13,50,AK,WY
5,month,DATE,0.00,312,2000-01-31,2026-09-30
6,value,DOUBLE,0.00,248607,46293.54771693069,1591642.0777961127


In [5]:
profile(
    "zillow_metro_zori",
    ["RegionID", "month"],
    "month",
    "Zillow metro ID + name, no CBSA code",
)

zillow_metro_zori: 53,977 rows | key (RegionID, month): 0 duplicated | month: 2015-01-31 to 2026-09-30 | geography: Zillow metro ID + name, no CBSA code


,column_name,column_type,null_percentage,approx_unique,min,max
0,RegionID,BIGINT,0.00,747,102001,845169
1,SizeRank,BIGINT,0.00,748,0,922
2,RegionName,VARCHAR,0.00,738,"Aberdeen, SD","Zanesville, OH"
3,RegionType,VARCHAR,0.00,2,country,msa
4,StateName,VARCHAR,0.26,50,AK,WY
5,month,DATE,0.00,141,2015-01-31,2026-09-30
6,value,DOUBLE,0.00,56585,526.9858831144877,4864.95969881673


Month labels are end-of-month dates. Share of ZIP × month cells with a value, by year (each file against its own ZIP list): ZORI is much sparser than ZHVI.

In [6]:
q("""
with f as (
    select 'zhvi' src, year(month) yr, count(*) n, count(distinct month) n_months, (select count(*) from zillow_zip_zhvi_wide) regions
    from zillow_zip_zhvi where month >= '2015-01-01' group by all
    union all
    select 'zori', year(month), count(*), count(distinct month), (select count(*) from zillow_zip_zori_wide)
    from zillow_zip_zori where month >= '2015-01-01' group by all)
pivot (select src, yr, round(100 * n / (n_months * regions), 1) pct_filled from f) on src using any_value(pct_filled) order by yr
""").T

,0,1,2,3,4,5,6,7,8,9,10,11
yr,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0,2025.0,2026.0
zhvi,83.1,88.4,90.0,91.1,93.5,94.0,95.1,97.8,98.5,100.0,100.0,100.0
zori,11.1,15.5,19.7,23.5,26.3,28.2,34.9,47.8,57.5,65.1,73.3,88.2


Zillow metro names for the study metros. Zillow keys metros by its own RegionID and a short name, so the CBSA mapping must be hand-written (Phase 3).

In [7]:
q("""
select distinct RegionID, RegionName, SizeRank from zillow_metro_zhvi
where regexp_matches(RegionName, '^(Nashville|Atlanta|Orlando|Seattle|Austin|Phoenix|Raleigh|Charlotte|Tampa|Dallas|Denver), (TN|GA|FL|WA|TX|AZ|NC|CO)$')
order by SizeRank
""")

,RegionID,RegionName,SizeRank
0,394514,"Dallas, TX",4
1,394347,"Atlanta, GA",9
2,394976,"Phoenix, AZ",11
3,395078,"Seattle, WA",15
4,395148,"Tampa, FL",18
5,394530,"Denver, CO",19
6,394943,"Orlando, FL",22
7,394458,"Charlotte, NC",23
8,394355,"Austin, TX",29
9,394902,"Nashville, TN",37


In [8]:
# ZIP files name the metro by its pre-2023 CBSA title (e.g. Denver-Aurora-Lakewood), not by code
q(f"""
select Metro, count(distinct RegionID) zips from read_csv('{S}/zillow/zip_zhvi.csv', types={{'RegionName': 'VARCHAR'}})
where regexp_matches(Metro, 'Nashville|Atlanta|Orlando|Seattle|Austin-|Phoenix|Raleigh|Charlotte|Tampa|Dallas|Denver')
group by all order by 1
""")

,Metro,zips
0,"Atlanta-Sandy Springs-Alpharetta, GA",203
1,"Austin-Round Rock-Georgetown, TX",87
2,"Charlotte-Concord-Gastonia, NC-SC",105
3,"Charlottesville, VA",37
4,"Dallas-Fort Worth-Arlington, TX",254
5,"Denver-Aurora-Lakewood, CO",126
6,"Nashville-Davidson--Murfreesboro--Franklin, TN",102
7,"Orlando-Kissimmee-Sanford, FL",90
8,"Phoenix-Mesa-Chandler, AZ",149
9,"Raleigh-Cary, NC",47


## 2. Redfin Data Center (market tracker)

Long files, one row per region, period, property type and seasonal-adjustment flag. The ZIP file (1.5 GB gzipped)
is converted to Parquet once so it is not re-decompressed on every run. Missing values are the text `NA`.

In [9]:
zip_pq = Path(S) / "redfin" / "zip_market_tracker.parquet"
if not zip_pq.exists():
    con.sql(
        f"copy (select * from read_csv('{S}/redfin/zip_market_tracker.tsv.gz', delim='\t', nullstr='NA', types={{'REGION': 'VARCHAR'}})) to '{zip_pq.as_posix()}'"
    )
con.sql(
    f"create or replace table redfin_metro as select * from read_csv('{S}/redfin/metro_market_tracker.tsv.gz', delim='\t', nullstr='NA')"
)
con.sql(f"create or replace view redfin_zip as select * from '{zip_pq.as_posix()}'")
RF_KEY = ["TABLE_ID", "PERIOD_BEGIN", "PROPERTY_TYPE_ID", "IS_SEASONALLY_ADJUSTED"]
profile(
    "redfin_metro",
    RF_KEY,
    "PERIOD_BEGIN",
    "Redfin metro (TABLE_ID = CBSA or metro division code)",
)

redfin_metro: 579,544 rows | key (TABLE_ID, PERIOD_BEGIN, PROPERTY_TYPE_ID, IS_SEASONALLY_ADJUSTED): 0 duplicated | PERIOD_BEGIN: 2012-01-01 to 2026-05-01 | geography: Redfin metro (TABLE_ID = CBSA or metro division code)


,column_name,column_type,null_percentage,approx_unique,min,max
0,PERIOD_BEGIN,DATE,0.00,175,2012-01-01,2026-05-01
1,PERIOD_END,DATE,0.00,173,2012-01-31,2026-05-31
2,PERIOD_DURATION,BIGINT,0.00,1,30,30
3,REGION_TYPE,VARCHAR,0.00,1,metro,metro
4,REGION_TYPE_ID,BIGINT,0.00,1,-2,-2
5,TABLE_ID,BIGINT,0.00,931,10100,49820
6,IS_SEASONALLY_ADJUSTED,BOOLEAN,0.00,2,false,true
7,REGION,VARCHAR,0.00,912,"Aberdeen, SD metro area","Zapata, TX metro area"
8,CITY,VARCHAR,0.00,1,,
9,STATE,VARCHAR,0.00,1,,


In [10]:
profile(
    "redfin_zip",
    RF_KEY,
    "PERIOD_BEGIN",
    "ZIP, parsed from REGION 'Zip Code: NNNNN'; TABLE_ID is Redfin's own",
)

redfin_zip: 9,725,026 rows | key (TABLE_ID, PERIOD_BEGIN, PROPERTY_TYPE_ID, IS_SEASONALLY_ADJUSTED): 0 duplicated | PERIOD_BEGIN: 2012-01-01 to 2026-03-01 | geography: ZIP, parsed from REGION 'Zip Code: NNNNN'; TABLE_ID is Redfin's own


,column_name,column_type,null_percentage,approx_unique,min,max
0,PERIOD_BEGIN,DATE,0.00,173,2012-01-01,2026-03-01
1,PERIOD_END,DATE,0.00,170,2012-03-31,2026-05-31
2,PERIOD_DURATION,BIGINT,0.00,1,90,90
3,REGION_TYPE,VARCHAR,0.00,1,zip code,zip code
4,REGION_TYPE_ID,BIGINT,0.00,1,2,2
5,TABLE_ID,BIGINT,0.00,25630,1,42122
6,IS_SEASONALLY_ADJUSTED,BOOLEAN,0.00,1,false,false
7,REGION,VARCHAR,0.00,26355,Zip Code: 00501,Zip Code: 99901
8,CITY,VARCHAR,0.00,1,,
9,STATE,VARCHAR,0.00,52,Alabama,Wyoming


Period length, seasonal adjustment and last update. Metro rows are calendar months; ZIP rows are rolling 90-day windows, unadjusted only.

In [11]:
pd.concat(
    [
        q(
            f"select '{v}' src, PERIOD_DURATION, IS_SEASONALLY_ADJUSTED, count(*) n, max(PERIOD_BEGIN) latest, max(LAST_UPDATED) last_updated from {v} group by all"
        )
        for v in ("redfin_metro", "redfin_zip")
    ]
)

,src,PERIOD_DURATION,IS_SEASONALLY_ADJUSTED,n,latest,last_updated
0,redfin_metro,30,True,33295,2026-05-01,2026-06-02 00:00:00.000 Z
1,redfin_metro,30,False,546249,2026-05-01,2026-06-02 14:33:24.470 Z
0,redfin_zip,90,False,9725026,2026-03-01,2026-06-02 14:33:24.470 Z


In [12]:
# ZIP rows never carry months of supply or price drops; seasonally adjusted metro counts can go negative
q("""select 'redfin_zip' src, count(MONTHS_OF_SUPPLY) months_of_supply, count(PRICE_DROPS) price_drops, count(INVENTORY) inventory,
            count(*) filter (INVENTORY < 0) negative_inventory, count(*) n from redfin_zip
     union all
     select 'redfin_metro sa', count(MONTHS_OF_SUPPLY), count(PRICE_DROPS), count(INVENTORY), count(*) filter (INVENTORY < 0), count(*)
     from redfin_metro where IS_SEASONALLY_ADJUSTED""")

,src,months_of_supply,price_drops,inventory,negative_inventory,n
0,redfin_zip,0,0,8698600,0,9725026
1,redfin_metro sa,33165,31583,32113,5,33295


In [13]:
# Study metros as Redfin keys them: Dallas-Fort Worth (19124 + 23104) and Seattle (42644 + 45104) appear only as
# pre-2023 metro divisions, so their metro values are medians that cannot simply be summed
q("""
select distinct TABLE_ID, REGION from redfin_metro
where regexp_matches(REGION, '^(Nashville|Atlanta|Orlando|Seattle|Tacoma|Austin|Phoenix|Raleigh|Charlotte|Tampa|Dallas|Fort Worth|Denver), (TN|GA|FL|WA|TX|AZ|NC|CO) metro area')
order by 2
""")

,TABLE_ID,REGION
0,12060,"Atlanta, GA metro area"
1,12420,"Austin, TX metro area"
2,16740,"Charlotte, NC metro area"
3,19124,"Dallas, TX metro area"
4,19740,"Denver, CO metro area"
5,23104,"Fort Worth, TX metro area"
6,34980,"Nashville, TN metro area"
7,36740,"Orlando, FL metro area"
8,38060,"Phoenix, AZ metro area"
9,39580,"Raleigh, NC metro area"


## 3. Census Building Permits Survey

Comma-delimited text with two header rows and a blank line. Each structure type (1, 2, 3–4, 5+ units) has buildings,
units and value, first as estimates with imputation, then as reported only. Per the
[CBSA layout doc](https://www2.census.gov/econ/bps/Documentation/cbsaasc.pdf) CBSA value is in **thousands** of dollars;
county value per unit (below) shows the county file is in **dollars**.

In [14]:
groups = [f"{t}{s}" for s in ("", "_rep") for t in ("u1", "u2", "u34", "u5p")]
measures = [f"{g}_{m}" for g in groups for m in ("bldgs", "units", "value")]
cbsa_cols = [
    "survey_date",
    "csa_code",
    "cbsa_code",
    "header_code",
    "cbsa_name",
] + measures
co_cols = [
    "survey_date",
    "state_fips",
    "county_fips",
    "region_code",
    "division_code",
    "county_name",
] + measures
types = {m: "BIGINT" for m in measures}
con.sql(f"""create or replace view bps_cbsa as select * replace (trim(cbsa_code) as cbsa_code, trim(cbsa_name) as cbsa_name)
           from read_csv('{S}/bps/cbsa2608c.txt', skip=3, header=false, encoding='latin-1', names={cbsa_cols}, all_varchar=true, types={types})""")
con.sql(f"""create or replace view bps_county as select * replace (trim(county_name) as county_name)
           from read_csv('{S}/bps/co2608c.txt', skip=3, header=false, encoding='latin-1', names={co_cols}, all_varchar=true, types={types})""")
profile(
    "bps_cbsa",
    ["survey_date", "cbsa_code"],
    "survey_date",
    "CBSA (5-digit); header_code 2/4 metro, 5 micro",
)

bps_cbsa: 921 rows | key (survey_date, cbsa_code): 0 duplicated | survey_date: 202608 to 202608 | geography: CBSA (5-digit); header_code 2/4 metro, 5 micro


,column_name,column_type,null_percentage,approx_unique,min,max
0,survey_date,VARCHAR,0.0,1,202608,202608
1,csa_code,VARCHAR,0.0,191,101,999
2,cbsa_code,VARCHAR,0.0,914,10100,49780
3,header_code,VARCHAR,0.0,3,2,5
4,cbsa_name,VARCHAR,0.0,893,Aberdeen SD,Zanesville OH
5,u1_bldgs,BIGINT,0.0,209,0,3146
6,u1_units,BIGINT,0.0,209,0,3146
7,u1_value,BIGINT,0.0,821,0,990994
8,u2_bldgs,BIGINT,0.0,29,0,106
9,u2_units,BIGINT,0.0,29,0,212


In [15]:
profile(
    "bps_county",
    ["survey_date", "state_fips", "county_fips"],
    "survey_date",
    "County FIPS (2 + 3 digits)",
)

bps_county: 3,018 rows | key (survey_date, state_fips, county_fips): 0 duplicated | survey_date: 202608 to 202608 | geography: County FIPS (2 + 3 digits)


,column_name,column_type,null_percentage,approx_unique,min,max
0,survey_date,VARCHAR,0.0,1,202608,202608
1,state_fips,VARCHAR,0.0,52,01,56
2,county_fips,VARCHAR,0.0,304,000,840
3,region_code,VARCHAR,0.0,4,1,4
4,division_code,VARCHAR,0.0,9,1,9
5,county_name,VARCHAR,0.0,1799,Abbeville County,Zavala County
6,u1_bldgs,BIGINT,0.0,224,0,1034
7,u1_units,BIGINT,0.0,224,0,1034
8,u1_value,BIGINT,0.0,2081,0,421931928
9,u2_bldgs,BIGINT,0.0,29,0,104


In [16]:
# header_code: 2 = metro in a CSA, 4 = metro not in a CSA, 5 = micro. Names are padded and use two spaces before the state.
display(q("select header_code, count(*) n from bps_cbsa group by all order by 1"))
display(
    q(
        f"select cbsa_code, cbsa_name, header_code, u1_units, u5p_units from bps_cbsa where cbsa_code in {tuple(CBSAS)} order by 1"
    )
)
q("""select 'cbsa ($ thousands)' file, round(sum(u1_value) / sum(u1_units)) value_per_1unit from bps_cbsa
     union all select 'county ($)', round(sum(u1_value) / sum(u1_units)) from bps_county""")

,header_code,n
0,2,275
1,4,112
2,5,534


,cbsa_code,cbsa_name,header_code,u1_units,u5p_units
0,12060,Atlanta-Sandy Springs-Roswell GA,2,1787,875
1,12420,Austin-Round Rock-San Marcos TX,4,1147,730
2,16740,Charlotte-Concord-Gastonia NC-SC,2,1020,608
3,19100,Dallas-Fort Worth-Arlington TX,2,3146,1190
4,19740,Denver-Aurora-Centennial CO,2,563,103
5,34980,Nashville-Davidson--Murfreesboro--Franklin TN,2,1209,284
6,36740,Orlando-Kissimmee-Sanford FL,2,1120,328
7,38060,Phoenix-Mesa-Chandler AZ,2,1468,2052
8,39580,Raleigh-Cary NC,2,979,412
9,42660,Seattle-Tacoma-Bellevue WA,2,415,473


,file,value_per_1unit
0,cbsa ($ thousands),340.0
1,county ($),339102.0


## 4. Census ACS 5-year (2020–2024)

API JSON: an array of arrays, first row is the header. Variables: `B19013_001E` median household income,
`B01003_001E` total population, `B25064_001E` median gross rent, `B25071_001E` median gross rent as % of income.
Values arrive as text; negative sentinels (for example −666666666) mean "not available".

In [17]:
ACS_NAMES = {
    "B19013_001E": "median_hh_income",
    "B01003_001E": "population",
    "B25064_001E": "median_gross_rent",
    "B25071_001E": "rent_pct_income",
}
for geo, idcols in {
    "zcta": ["zip code tabulation area"],
    "county": ["state", "county"],
    "cbsa": ["metropolitan statistical area/micropolitan statistical area"],
}.items():
    rows = json.loads((Path(S) / "acs" / f"acs5_2024_{geo}.json").read_text())
    df = pd.DataFrame(rows[1:], columns=rows[0]).rename(columns=ACS_NAMES)
    df["geo_id"] = df[idcols].agg("".join, axis=1)
    for c in ACS_NAMES.values():
        df[c] = pd.to_numeric(df[c])
    con.register(f"acs_{geo}_df", df[["geo_id", "NAME", *ACS_NAMES.values()]])
    con.sql(f"create or replace view acs_{geo} as select * from acs_{geo}_df")
profile("acs_zcta", ["geo_id"], None, "ZCTA (5-char text)", "ACS 5-year 2020-2024")

acs_zcta: 33,772 rows | key (geo_id): 0 duplicated | vintage: ACS 5-year 2020-2024 | geography: ZCTA (5-char text)


,column_name,column_type,null_percentage,approx_unique,min,max
0,geo_id,VARCHAR,0.0,34907,00601,99929
1,NAME,VARCHAR,0.0,31010,ZCTA5 00601,ZCTA5 99929
2,median_hh_income,BIGINT,0.0,21017,-666666666,250001
3,population,BIGINT,0.0,15931,0,140157
4,median_gross_rent,BIGINT,0.0,2477,-666666666,3501
5,rent_pct_income,DOUBLE,0.0,406,-666666666.0,51.0


In [18]:
profile(
    "acs_county",
    ["geo_id"],
    None,
    "County FIPS (state + county = 5 chars)",
    "ACS 5-year 2020-2024",
)

acs_county: 3,222 rows | key (geo_id): 0 duplicated | vintage: ACS 5-year 2020-2024 | geography: County FIPS (state + county = 5 chars)


,column_name,column_type,null_percentage,approx_unique,min,max
0,geo_id,VARCHAR,0.0,3167,01001,72153
1,NAME,VARCHAR,0.0,3168,"Abbeville County, South Carolina","Ziebach County, South Dakota"
2,median_hh_income,BIGINT,0.0,3064,-666666666,181765
3,population,BIGINT,0.0,3065,33,9808667
4,median_gross_rent,BIGINT,0.0,1020,-666666666,2922
5,rent_pct_income,DOUBLE,0.0,272,-666666666.0,51.0


In [19]:
profile("acs_cbsa", ["geo_id"], None, "CBSA (5-digit)", "ACS 5-year 2020-2024")

acs_cbsa: 935 rows | key (geo_id): 0 duplicated | vintage: ACS 5-year 2020-2024 | geography: CBSA (5-digit)


,column_name,column_type,null_percentage,approx_unique,min,max
0,geo_id,VARCHAR,0.0,924,10100,49820
1,NAME,VARCHAR,0.0,910,"Aberdeen, SD Micro Area","Zapata, TX Micro Area"
2,median_hh_income,BIGINT,0.0,932,18917,162111
3,population,BIGINT,0.0,917,12616,19798537
4,median_gross_rent,BIGINT,0.0,543,470,2840
5,rent_pct_income,DOUBLE,0.0,154,18.4,45.9


In [20]:
# Sentinel (negative) values per variable: these must become NULL in staging
pd.concat(
    [
        q(
            f"select '{g}' geo, "
            + ", ".join(f"count(*) filter ({c} < 0) {c}" for c in ACS_NAMES.values())
            + f", count(*) n from acs_{g}"
        )
        for g in ("zcta", "county", "cbsa")
    ]
)

,geo,median_hh_income,population,median_gross_rent,rent_pct_income,n
0,zcta,3225,0,8021,7440,33772
0,county,1,0,7,5,3222
0,cbsa,0,0,0,0,935


In [21]:
q(f"select * from acs_cbsa where geo_id in {tuple(CBSAS)} order by geo_id")

,geo_id,NAME,median_hh_income,population,median_gross_rent,rent_pct_income
0,12060,"Atlanta-Sandy Springs-Roswell, GA Metro Area",89724,6256377,1672,32.3
1,12420,"Austin-Round Rock-San Marcos, TX Metro Area",100431,2426592,1726,29.6
2,16740,"Charlotte-Concord-Gastonia, NC-SC Metro Area",83304,2768407,1473,29.5
3,19100,"Dallas-Fort Worth-Arlington, TX Metro Area",90275,7985590,1614,31.2
4,19740,"Denver-Aurora-Centennial, CO Metro Area",105762,3002721,1874,30.8
5,34980,"Nashville-Davidson--Murfreesboro--Franklin, TN...",85447,2079596,1528,30.7
6,36740,"Orlando-Kissimmee-Sanford, FL Metro Area",78533,2793746,1760,35.0
7,38060,"Phoenix-Mesa-Chandler, AZ Metro Area",88301,5028754,1702,31.2
8,39580,"Raleigh-Cary, NC Metro Area",100103,1487302,1573,29.3
9,42660,"Seattle-Tacoma-Bellevue, WA Metro Area",115177,4061920,2000,29.6


## 5. BLS QCEW (2026 Q1, all industries)

The "industry 10" slice: total, all industries, for every area and ownership. `area_fips` mixes county FIPS (`47037`),
state (`47000`), U.S. (`US000`), CSA (`CS400`) and MSA codes (`C3498` = `C` + first four digits of the CBSA).

In [22]:
con.sql(f"""create or replace view qcew as select * from read_csv('{S}/qcew/2026_q1_industry_10.csv',
           types={{'area_fips': 'VARCHAR', 'own_code': 'VARCHAR', 'industry_code': 'VARCHAR', 'agglvl_code': 'VARCHAR', 'size_code': 'VARCHAR'}})""")
profile(
    "qcew",
    [
        "area_fips",
        "own_code",
        "industry_code",
        "agglvl_code",
        "size_code",
        "year",
        "qtr",
    ],
    None,
    "County FIPS, state, MSA (C+4 digits), CSA",
    "2026 Q1",
)

qcew: 17,729 rows | key (area_fips, own_code, industry_code, agglvl_code, size_code, year, qtr): 0 duplicated | vintage: 2026 Q1 | geography: County FIPS, state, MSA (C+4 digits), CSA


,column_name,column_type,null_percentage,approx_unique,min,max
0,area_fips,VARCHAR,0.00,4426,01000,USNMS
1,own_code,VARCHAR,0.00,7,0,9
2,industry_code,VARCHAR,0.00,1,10,10
3,agglvl_code,VARCHAR,0.00,15,10,96
4,size_code,VARCHAR,0.00,1,0,0
5,year,BIGINT,0.00,1,2026,2026
6,qtr,BIGINT,0.00,1,1,1
7,disclosure_code,VARCHAR,98.04,1,N,N
8,qtrly_estabs,BIGINT,0.00,3731,1,12381750
9,month1_emplvl,BIGINT,0.00,8306,0,154172010


In [23]:
# agglvl 70/71 = county total / county by ownership, 40/41 = MSA; own_code 0 = all, 5 = private
display(
    q(
        "select agglvl_code, own_code, count(*) n, count(*) filter (disclosure_code = 'N') suppressed from qcew group by all order by 1, 2"
    )
)
q(f"""select area_fips, own_code, month3_emplvl, avg_wkly_wage from qcew
      where agglvl_code = '40' and own_code = '0' and area_fips in {tuple("C" + c[:4] for c in CBSAS)} order by 1""")

,agglvl_code,own_code,n,suppressed
0,10,0,1,0
1,11,1,1,0
2,11,2,1,0
3,11,3,1,0
4,11,5,1,0
5,30,0,184,0
6,40,0,393,0
7,50,0,53,0
8,51,1,53,0
9,51,2,53,3


,area_fips,own_code,month3_emplvl,avg_wkly_wage
0,C1206,0,2857128,1734
1,C1242,0,1308022,1957
2,C1674,0,1363220,1785
3,C1910,0,4034579,1785
4,C1974,0,1578155,1990
5,C3498,0,1081446,1626
6,C3674,0,1409855,1362
7,C3806,0,2378551,1557
8,C3958,0,748509,1621
9,C4266,0,2056852,2388


## 6. FRED (mortgage rate, CPI, metro unemployment)

`MORTGAGE30US` weekly, `CPIAUCSL` monthly seasonally adjusted, `LAUMT473498000000003` Nashville MSA unemployment rate
(monthly, not adjusted). Metro LAUS IDs follow `LAUMT` + state FIPS + CBSA + `000000003`.

In [24]:
con.sql(
    "create or replace view fred as "
    + " union all ".join(
        f"select '{s}' series_id, o.date::date date, nullif(o.value, '.')::double value, o.value = '.' is_missing "
        f"from (select unnest(observations) o from read_json('{S}/fred/{s}.json'))"
        for s in ("MORTGAGE30US", "CPIAUCSL", "LAUMT473498000000003")
    )
)
profile("fred", ["series_id", "date"], "date", "National; metro via LAUS series ID")

fred: 4,297 rows | key (series_id, date): 0 duplicated | date: 1947-01-01 to 2026-10-22 | geography: National; metro via LAUS series ID


,column_name,column_type,null_percentage,approx_unique,min,max
0,series_id,VARCHAR,0.00,3,CPIAUCSL,MORTGAGE30US
1,date,DATE,0.00,3667,1947-01-01,2026-10-22
2,value,DOUBLE,0.05,1877,2.2,336.298
3,is_missing,BOOLEAN,0.00,2,false,true


In [25]:
q(
    "select series_id, count(*) n, min(date), max(date), count(*) filter (is_missing) missing, min(value), max(value) from fred group by all"
)

,series_id,n,min(date),max(date),missing,"min(""value"")","max(""value"")"
0,LAUMT473498000000003,440,1990-01-01,2026-08-01,1,2.20,15.100
1,CPIAUCSL,957,1947-01-01,2026-09-01,1,21.48,336.298
2,MORTGAGE30US,2900,1971-04-02,2026-10-22,0,2.65,18.630


## 7. IRS SOI county migration (2022–2023)

`n1` = returns (households), `n2` = exemptions (people), `agi` = adjusted gross income in **thousands** of dollars.
In the inflow file `y2_*` is the destination and `y1_*` the origin; the outflow file swaps them. State codes 96–98 are
summary rows (total, U.S., same state...), 57–59 are foreign/other-state rollups; `-1` marks suppressed cells.

In [26]:
for f in ("inflow", "outflow"):
    con.sql(f"""create or replace view irs_{f} as select * from read_csv('{S}/irs/county{f}2223.csv', encoding='latin-1',
               types={{'y1_statefips': 'VARCHAR', 'y1_countyfips': 'VARCHAR', 'y2_statefips': 'VARCHAR', 'y2_countyfips': 'VARCHAR'}})""")
IRS_KEY = ["y2_statefips", "y2_countyfips", "y1_statefips", "y1_countyfips"]
profile(
    "irs_inflow",
    IRS_KEY,
    None,
    "County FIPS pairs, origin y1 to destination y2",
    "tax years 2022 to 2023",
)

irs_inflow: 90,048 rows | key (y2_statefips, y2_countyfips, y1_statefips, y1_countyfips): 0 duplicated | vintage: tax years 2022 to 2023 | geography: County FIPS pairs, origin y1 to destination y2


,column_name,column_type,null_percentage,approx_unique,min,max
0,y2_statefips,VARCHAR,0.0,52,01,56
1,y2_countyfips,VARCHAR,0.0,329,001,840
2,y1_statefips,VARCHAR,0.0,59,01,98
3,y1_countyfips,VARCHAR,0.0,329,000,840
4,y1_state,VARCHAR,0.0,53,AK,WY
5,y1_countyname,VARCHAR,0.0,10753,Abbeville County,Ziebach County Non-migrants
6,n1,BIGINT,0.0,6257,-1,3930114
7,n2,BIGINT,0.0,7829,-1,7351115
8,agi,BIGINT,0.0,28525,-1,414373680


In [27]:
profile(
    "irs_outflow",
    IRS_KEY,
    None,
    "County FIPS pairs, origin y1 to destination y2",
    "tax years 2022 to 2023",
)

irs_outflow: 90,358 rows | key (y2_statefips, y2_countyfips, y1_statefips, y1_countyfips): 0 duplicated | vintage: tax years 2022 to 2023 | geography: County FIPS pairs, origin y1 to destination y2


,column_name,column_type,null_percentage,approx_unique,min,max
0,y1_statefips,VARCHAR,0.0,52,01,56
1,y1_countyfips,VARCHAR,0.0,329,001,840
2,y2_statefips,VARCHAR,0.0,59,01,98
3,y2_countyfips,VARCHAR,0.0,329,000,840
4,y2_state,VARCHAR,0.0,53,AK,WY
5,y2_countyname,VARCHAR,0.0,10783,Abbeville County,Ziebach County Total Migration-US and Foreign
6,n1,BIGINT,0.0,6423,-1,3930114
7,n2,BIGINT,0.0,8047,-1,7351115
8,agi,BIGINT,0.0,29045,-100,414373680


In [28]:
q("""select case when y1_statefips::int >= 96 then 'summary 96-98' when y1_statefips::int >= 57 then 'other 57-59'
            else 'county to county' end row_kind, count(*) n, count(*) filter (n1 = -1) suppressed
     from irs_inflow group by all order by 1""")

,row_kind,n,suppressed
0,county to county,56757,0
1,other 57-59,20694,5818
2,summary 96-98,12597,0


## 8. HUD Fair Market Rents (FY2027, API)

Monthly rent benchmarks in dollars by bedroom count. Nashville is a Small Area FMR metro, so the API returns one row per ZIP.

In [29]:
nash = json.loads((Path(S) / "hud_fmr" / "fmr_2027_nashville.json").read_text())["data"]
print({k: v for k, v in nash.items() if k != "basicdata"})
con.register("hud_fmr_zip_df", pd.DataFrame(nash["basicdata"]))
con.sql("create or replace view hud_fmr_zip as select * from hud_fmr_zip_df")
profile(
    "hud_fmr_zip",
    ["zip_code"],
    None,
    "ZIP (Small Area FMR) within a HUD FMR area",
    "FY2027",
)

{'county_name': '', 'counties_msa': 'Cannon County, TN; Cheatham County, TN; Davidson County, TN; Dickson County, TN; Robertson County, TN; Rutherford County, TN; Sumner County, TN; Trousdale County, TN; Williamson County, TN; and Wilson County, TN', 'town_name': '', 'metro_status': '1.0', 'metro_name': 'Nashville-Davidson--Murfreesboro--Franklin, TN', 'area_name': 'Nashville-Davidson--Murfreesboro--Franklin, TN HUD Metro FMR Area', 'smallarea_status': '1', 'year': '2027'}
hud_fmr_zip: 126 rows | key (zip_code): 0 duplicated | vintage: FY2027 | geography: ZIP (Small Area FMR) within a HUD FMR area


,column_name,column_type,null_percentage,approx_unique,min,max
0,zip_code,VARCHAR,0.0,125,37010,MSA level
1,Efficiency,BIGINT,0.0,65,800,2380
2,One-Bedroom,BIGINT,0.0,67,800,2480
3,Two-Bedroom,BIGINT,0.0,67,1050,2700
4,Three-Bedroom,BIGINT,0.0,68,1250,3470
5,Four-Bedroom,BIGINT,0.0,70,1380,4200


In [30]:
tn = json.loads((Path(S) / "hud_fmr" / "fmr_2027_state_tn.json").read_text())["data"]
con.register("hud_fmr_county_df", pd.DataFrame(tn["counties"]))
con.sql("create or replace view hud_fmr_county as select * from hud_fmr_county_df")
profile(
    "hud_fmr_county",
    ["fips_code"],
    None,
    "County (10-char FIPS: state + county + 99999 suffix)",
    "FY2027",
)

hud_fmr_county: 95 rows | key (fips_code): 0 duplicated | vintage: FY2027 | geography: County (10-char FIPS: state + county + 99999 suffix)


,column_name,column_type,null_percentage,approx_unique,min,max
0,town_name,"""NULL""",100.0,0,NaN,NaN
1,county_name,VARCHAR,0.0,95,Anderson County,Wilson County
2,metro_name,VARCHAR,0.0,73,"Bedford County, TN","White County, TN"
3,fips_code,VARCHAR,0.0,96,4700199999,4718999999
4,Efficiency,BIGINT,0.0,37,693,1585
5,One-Bedroom,BIGINT,0.0,43,761,1650
6,Two-Bedroom,BIGINT,0.0,34,995,1798
7,Three-Bedroom,BIGINT,0.0,57,1186,2309
8,Four-Bedroom,BIGINT,0.0,49,1308,2799
9,FMR Percentile,BIGINT,0.0,1,40,40


## 9. HUD-USPS ZIP crosswalk (API)

One row per ZIP and target geography with the share of residential, business, other and total addresses.
A ZIP split across counties has several rows; its ratios should sum to 1.

In [31]:
for t in ("county", "cbsa"):
    con.sql(f"""create or replace view hud_zip_{t} as
               select data.year, data.quarter, unnest(data.results, recursive := true)
               from read_json('{S}/hud_crosswalk/zip_{t}.json', maximum_object_size = 64000000)""")
profile(
    "hud_zip_county", ["zip", "geoid"], None, "ZIP to county FIPS (5 chars)", "2026 Q2"
)

hud_zip_county: 54,570 rows | key (zip, geoid): 0 duplicated | vintage: 2026 Q2 | geography: ZIP to county FIPS (5 chars)


,column_name,column_type,null_percentage,approx_unique,min,max
0,year,VARCHAR,0.0,1,2026,2026
1,quarter,VARCHAR,0.0,1,2,2
2,zip,VARCHAR,0.0,40569,00501,99950
3,geoid,VARCHAR,0.0,3174,01001,78030
4,city,VARCHAR,0.0,19303,AARONSBURG,ZWOLLE
5,state,VARCHAR,0.0,59,AK,WY
6,res_ratio,DOUBLE,0.0,23801,0.0,1.0
7,bus_ratio,DOUBLE,0.0,7209,0.0,1.0
8,oth_ratio,DOUBLE,0.0,3532,0.0,1.0
9,tot_ratio,DOUBLE,0.0,24146,2.7071658680527356e-05,1.0


In [32]:
profile(
    "hud_zip_cbsa",
    ["zip", "geoid"],
    None,
    "ZIP to CBSA (5 digits; 99999 = outside any CBSA)",
    "2026 Q2",
)

hud_zip_cbsa: 47,629 rows | key (zip, geoid): 0 duplicated | vintage: 2026 Q2 | geography: ZIP to CBSA (5 digits; 99999 = outside any CBSA)


,column_name,column_type,null_percentage,approx_unique,min,max
0,year,VARCHAR,0.0,1,2026,2026
1,quarter,VARCHAR,0.0,1,2,2
2,zip,VARCHAR,0.0,40451,00501,99950
3,geoid,VARCHAR,0.0,924,10100,99999
4,city,VARCHAR,0.0,19303,AARONSBURG,ZWOLLE
5,state,VARCHAR,0.0,54,AK,WY
6,res_ratio,DOUBLE,0.0,14383,0.0,1.0
7,bus_ratio,DOUBLE,0.0,3780,0.0,1.0
8,oth_ratio,DOUBLE,0.0,1705,0.0,1.0
9,tot_ratio,DOUBLE,0.0,13988,2.7071658680527356e-05,1.0


In [33]:
q("""select any_value(yr) yr, any_value(qtr) qtr, count(*) zips,
            count(*) filter (n > 1) split_zips,
            count(*) filter (abs(res_sum - 1) > 0.01 and res_sum > 0) res_not_1,
            count(*) filter (res_sum = 0) no_residential
     from (select zip, any_value("year") yr, any_value("quarter") qtr, count(*) n, sum(res_ratio) res_sum
           from hud_zip_county group by zip)""")

,yr,qtr,zips,split_zips,res_not_1,no_residential
0,2026,2,39484,11379,0,3549


In [34]:
q(
    f"select geoid cbsa_code, count(distinct zip) zips from hud_zip_cbsa where geoid in {tuple(CBSAS)} group by all order by 1"
)

,cbsa_code,zips
0,12060,327
1,12420,139
2,16740,185
3,19100,388
4,19740,195
5,34980,164
6,36740,154
7,38060,223
8,39580,84
9,42660,230


## 10. Census CBSA delineation (July 2023)

Excel: two title rows, header on row 3, notes after the data. One row per county with its CBSA, metro division and CSA.

In [35]:
con.sql(f"""create or replace view cbsa_delineation as
           select * from read_xlsx('{S}/cbsa_delineation/list1_2023.xlsx', range = 'A3:L5000', all_varchar = true)
           where regexp_full_match("CBSA Code", '\\d{{5}}')""")
profile(
    "cbsa_delineation",
    ['"FIPS State Code"', '"FIPS County Code"'],
    None,
    "County FIPS to CBSA, metro division, CSA",
    "July 2023",
)

cbsa_delineation: 1,915 rows | key ("FIPS State Code", "FIPS County Code"): 0 duplicated | vintage: July 2023 | geography: County FIPS to CBSA, metro division, CSA


,column_name,column_type,null_percentage,approx_unique,min,max
0,CBSA Code,VARCHAR,0.00,924,10100,49820
1,Metropolitan Division Code,VARCHAR,92.74,38,11244,48864
2,CSA Code,VARCHAR,29.97,193,101,566
3,CBSA Title,VARCHAR,0.00,995,"Aberdeen, SD","Zapata, TX"
4,Metropolitan/Micropolitan Statistical Area,VARCHAR,0.00,2,Metropolitan Statistical Area,Micropolitan Statistical Area
5,Metropolitan Division Title,VARCHAR,92.74,36,"Anaheim-Santa Ana-Irvine, CA","Wilmington, DE-MD-NJ"
6,CSA Title,VARCHAR,29.97,189,"Abilene-Sweetwater, TX","Youngstown-Warren-Salem, OH"
7,County/County Equivalent,VARCHAR,0.00,1312,Abbeville County,Zapata County
8,State Name,VARCHAR,0.00,53,Alabama,Wyoming
9,FIPS State Code,VARCHAR,0.00,53,01,72


In [36]:
q(f"""select "CBSA Code", any_value("CBSA Title") title, count(*) counties,
             string_agg(distinct "Metropolitan Division Code", ', ') divisions
      from cbsa_delineation where "CBSA Code" in {tuple(CBSAS)} group by all order by 1""")

,CBSA Code,title,counties,divisions
0,12060,"Atlanta-Sandy Springs-Roswell, GA",29,"31924, 12054"
1,12420,"Austin-Round Rock-San Marcos, TX",5,NaN
2,16740,"Charlotte-Concord-Gastonia, NC-SC",11,NaN
3,19100,"Dallas-Fort Worth-Arlington, TX",11,"19124, 23104"
4,19740,"Denver-Aurora-Centennial, CO",10,NaN
5,34980,"Nashville-Davidson--Murfreesboro--Franklin, TN",14,NaN
6,36740,"Orlando-Kissimmee-Sanford, FL",4,NaN
7,38060,"Phoenix-Mesa-Chandler, AZ",2,NaN
8,39580,"Raleigh-Cary, NC",3,NaN
9,42660,"Seattle-Tacoma-Bellevue, WA",3,"21794, 45104, 42644"


## 11. Census cartographic boundaries (1:500k)

Shapefiles inside zip archives, read with DuckDB spatial. CBSA and county are 2024 vintage; ZCTAs exist only for the 2020 census.

In [37]:
for view, f in {
    "geo_cbsa": "cb_2024_us_cbsa_500k",
    "geo_county": "cb_2024_us_county_500k",
    "geo_zcta": "cb_2020_us_zcta520_500k",
}.items():
    con.sql(
        f"create or replace table {view}_shp as select * from st_read('/vsizip/{S}/boundaries/{f}.zip/{f}.shp')"
    )
    con.sql(
        f"create or replace view {view} as select * exclude (geom), st_geometrytype(geom)::varchar geom_type from {view}_shp"
    )
profile("geo_cbsa", ["GEOID"], None, "CBSA (GEOID, 5 digits)", "2024")

geo_cbsa: 935 rows | key (GEOID): 0 duplicated | vintage: 2024 | geography: CBSA (GEOID, 5 digits)


,column_name,column_type,null_percentage,approx_unique,min,max
0,OGC_FID,BIGINT,0.00,926,0,934
1,CSAFP,VARCHAR,37.75,193,101,566
2,CBSAFP,VARCHAR,0.00,924,10100,49820
3,GEOIDFQ,VARCHAR,0.00,945,310M700US10100,310M700US49820
4,GEOID,VARCHAR,0.00,924,10100,49820
5,NAME,VARCHAR,0.00,995,"Aberdeen, SD","Zapata, TX"
6,NAMELSAD,VARCHAR,0.00,910,"Aberdeen, SD Micro Area","Zapata, TX Micro Area"
7,LSAD,VARCHAR,0.00,2,M1,M2
8,ALAND,BIGINT,0.00,918,119637317,70648193821
9,AWATER,BIGINT,0.00,931,208105,15329022675


In [38]:
profile("geo_county", ["GEOID"], None, "County FIPS (GEOID, 5 chars)", "2024")

geo_county: 3,235 rows | key (GEOID): 0 duplicated | vintage: 2024 | geography: County FIPS (GEOID, 5 chars)


,column_name,column_type,null_percentage,approx_unique,min,max
0,OGC_FID,BIGINT,0.0,3209,0,3234
1,STATEFP,VARCHAR,0.0,58,01,78
2,COUNTYFP,VARCHAR,0.0,333,001,840
3,COUNTYNS,VARCHAR,0.0,3265,00023901,02830254
4,GEOIDFQ,VARCHAR,0.0,3278,0500000US01001,0500000US78030
5,GEOID,VARCHAR,0.0,3176,01001,78030
6,NAME,VARCHAR,0.0,1946,Abbeville,Ziebach
7,NAMELSAD,VARCHAR,0.0,2005,Abbeville County,Ziebach County
8,STUSPS,VARCHAR,0.0,55,AK,WY
9,STATE_NAME,VARCHAR,0.0,58,Alabama,Wyoming


In [39]:
profile("geo_zcta", ["GEOID20"], None, "ZCTA (GEOID20, 5 chars)", "2020")

geo_zcta: 33,791 rows | key (GEOID20): 0 duplicated | vintage: 2020 | geography: ZCTA (GEOID20, 5 chars)


,column_name,column_type,null_percentage,approx_unique,min,max
0,OGC_FID,BIGINT,0.0,33064,0,33790
1,ZCTA5CE20,VARCHAR,0.0,34925,00601,99929
2,AFFGEOID20,VARCHAR,0.0,33500,860Z200US00601,860Z200US99929
3,GEOID20,VARCHAR,0.0,34925,00601,99929
4,NAME20,VARCHAR,0.0,34925,00601,99929
5,LSAD20,VARCHAR,0.0,1,Z5,Z5
6,ALAND20,BIGINT,0.0,32823,3551,35426845132
7,AWATER20,BIGINT,0.0,28499,0,3695341658
8,geom_type,VARCHAR,0.0,2,MULTIPOLYGON,POLYGON


## Summary

One row per profiled table. `days_behind` is how far the latest period sits behind the day this notebook ran.

In [40]:
summary = pd.DataFrame(SUMMARY)
summary["days_behind"] = [
    (
        TODAY
        - pd.Timestamp(
            str(d)[:4] + "-" + str(d)[4:6] + "-01" if str(d).isdigit() else d
        ).date()
    ).days
    if d is not None
    else None
    for d in summary["date_to"]
]
summary

,source,rows,key,duplicate_keys,date_from,date_to,vintage,geography,days_behind
0,zillow_zip_zhvi,6508601,"RegionID, month",0,2000-01-31,2026-09-30,NaN,"ZIP (RegionName, 5-char text)",27.0
1,zillow_zip_zori,469420,"RegionID, month",0,2015-01-31,2026-09-30,NaN,"ZIP (RegionName, 5-char text)",27.0
2,zillow_metro_zhvi,238029,"RegionID, month",0,2000-01-31,2026-09-30,NaN,"Zillow metro ID + name, no CBSA code",27.0
3,zillow_metro_zori,53977,"RegionID, month",0,2015-01-31,2026-09-30,NaN,"Zillow metro ID + name, no CBSA code",27.0
4,redfin_metro,579544,"TABLE_ID, PERIOD_BEGIN, PROPERTY_TYPE_ID, IS_S...",0,2012-01-01,2026-05-01,NaN,Redfin metro (TABLE_ID = CBSA or metro divisio...,179.0
5,redfin_zip,9725026,"TABLE_ID, PERIOD_BEGIN, PROPERTY_TYPE_ID, IS_S...",0,2012-01-01,2026-03-01,NaN,"ZIP, parsed from REGION 'Zip Code: NNNNN'; TAB...",240.0
6,bps_cbsa,921,"survey_date, cbsa_code",0,202608,202608,NaN,"CBSA (5-digit); header_code 2/4 metro, 5 micro",87.0
7,bps_county,3018,"survey_date, state_fips, county_fips",0,202608,202608,NaN,County FIPS (2 + 3 digits),87.0
8,acs_zcta,33772,geo_id,0,None,None,ACS 5-year 2020-2024,ZCTA (5-char text),NaN
9,acs_county,3222,geo_id,0,None,None,ACS 5-year 2020-2024,County FIPS (state + county = 5 chars),NaN
